# Gram anchoring for SDPO: Colab runner

*GenAI disclosure: Claude (Anthropic) generated this notebook. Brian Onieal reviewed it.*

This notebook runs the team's experiment code on a Colab GPU. Run the cells in order.

1. **Setup** (cells 1 to 4): check the GPU, connect Google Drive, install the code, run the tests.
2. **Timing test** (cells 5 and 6): one training step with thinking on and one with thinking off. It reports speed, memory, and how many responses got cut off. This decides the settings for the real runs.
3. **Real runs** (cells 7 to 9): locked until the team sets the values marked `TEAM` in `configs/`.

Runs are saved to Google Drive, so a dropped session loses at most a few steps. Rerun the same cell and it picks up where it stopped.

In [ ]:
# 1. Check the GPU. In the menu: Runtime > Change runtime type > L4 GPU.
import torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > L4 GPU."
name = torch.cuda.get_device_name(0)
memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name}, {memory_gb:.1f} GB")
if "T4" in name:
    print("This is a T4. It is slower and has less memory. Switch to an L4 if you can.")

In [ ]:
# 2. Connect Google Drive. Runs and checkpoints are saved there.
from google.colab import drive

drive.mount("/content/drive")
RUNS = "/content/drive/MyDrive/nlp-final-project-runs"
!mkdir -p "{RUNS}"
print("Saving runs to", RUNS)

In [ ]:
# 3. Get the code and install it.
BRANCH = "experiment-code"  # change to "main" after the branch is merged
%cd /content
!rm -rf NLP-Final-Project
!git clone -q -b {BRANCH} https://github.com/brianonieal/NLP-Final-Project
%cd /content/NLP-Final-Project
!pip install -q -e . pytest
# Colab ships an old torchao that makes peft refuse to build LoRA layers.
# This project does not use torchao, so remove it.
!pip uninstall -y -q torchao
!git log --oneline -1

In [ ]:
# 4. Run the test suite. Every test should pass before you spend GPU time.
!python -m pytest -q

## Timing test

Each cell below runs **one** SDPO step on 8 questions with 4 attempts each, up to 4,096 tokens per attempt. Expect several minutes per cell on an L4.

If a cell stops with `CUDA out of memory`, add `loss.train_max_tokens=1024` to its `--override` list and run it again.

In [ ]:
# 5a. Timing test, thinking ON (the proposal's setting).
timing = "python -m gram_sdpo.train --config configs/colab_timing.yaml --override"
get_ipython().system(
    f"{timing} model.enable_thinking=true train.output_dir={RUNS}/timing_thinking_on"
)

In [ ]:
# 5b. Timing test, thinking OFF.
get_ipython().system(
    f"{timing} model.enable_thinking=false train.output_dir={RUNS}/timing_thinking_off"
)

In [ ]:
# 6. Summary. Copy this cell's output back to Claude.
import json
import pathlib

import yaml

plan = yaml.safe_load(open("configs/sdpo.yaml"))
steps = plan["train"]["steps"]
questions = plan["train"]["questions_per_step"]
k = plan["eval"]["samples_per_problem"]
eval_problems = min(plan["eval"]["max_problems"], plan["data"]["heldout_size"]) + 30 + 40

for run in ["timing_thinking_on", "timing_thinking_off"]:
    log = pathlib.Path(RUNS) / run / "train_log.jsonl"
    if not log.exists():
        print(f"{run}: no log yet")
        continue
    m = json.loads(log.read_text().splitlines()[-1])
    speed = m["sampled_tokens_per_second"]
    train_hours = steps * questions * (m["seconds"] / 8) / 3600
    eval_hours = eval_problems * k * m["mean_response_tokens"] / speed / 3600
    print(f"== {run}")
    print(f"   accuracy on 32 attempts:        {m['train_accuracy']:.3f}")
    print(f"   attempts cut off by the limit:  {m['unfinished_fraction']:.0%}")
    print(f"   mean response tokens:           {m['mean_response_tokens']:.0f}")
    print(f"   attempts trained on:            {m['attempts_trained']:.0f} of 32")
    print(f"   sampling speed (tokens/sec):    {speed:.0f}")
    print(f"   seconds for the step:           {m['seconds']:.0f}")
    print(f"   peak GPU memory (GB):           {m['peak_gpu_gb']:.1f}")
    print(f"   projected: one {steps}-step training run   ~{train_hours:.1f} hours")
    print(f"   projected: one full evaluation             ~{eval_hours:.1f} hours")
print()
print("Projections use the placeholder settings in configs/sdpo.yaml.")
print("Check Colab's Resources panel for how many compute units per hour this GPU uses.")

## Real runs

Do not run these until the team has set the values marked `TEAM` in the config files and the timing test shows the plan fits your compute units. Then set `READY = True`.

Each cell is safe to rerun after a dropped session: training resumes from the last saved checkpoint, and evaluation skips checkpoints that already have results.

In [ ]:
# 7. Settings shared by every real run on an L4.
READY = False
L4 = f"rollout.batch_size=4 eval.batch_size=1 train.save_every=5 eval.probe_file={RUNS}/probes.json"


def run(config: str, name: str, extra: str = "", checkpoints: str = "final") -> None:
    """Train one run to completion, then evaluate it. Rerun to resume."""
    assert READY, "Set READY = True once the team has set the TEAM values."
    settings = f"--override {L4} train.output_dir={RUNS}/{name} {extra}"
    get_ipython().system(f"python -m gram_sdpo.train --config {config} --resume {settings}")
    get_ipython().system(
        f"python -m gram_sdpo.evaluate --config {config} --checkpoints {checkpoints} {settings}"
    )

In [ ]:
# 8. Experiment 1 and Experiment 2. Run one line at a time.
run("configs/sdpo.yaml", "sdpo", checkpoints="base all")  # Experiment 1: drift at every checkpoint
# run("configs/sdpo_kl.yaml", "sdpo_kl")
# run("configs/sdpo_gram.yaml", "sdpo_gram_w1", "loss.gram_weight=1.0")
# run("configs/sdpo_gram.yaml", "sdpo_gram_w0.1", "loss.gram_weight=0.1")

In [ ]:
# 9. One table for every run. Also saved as results.csv and results.md on Drive.
!python -m gram_sdpo.report --runs "{RUNS}"
print(open(f"{RUNS}/results.md").read())